# 02 — 2D Mordred Feature Generation

Bu v5 sürümünde bütün kod hücreleri eğitim amacıyla satır satır açıklanmıştır. Her çalıştırılabilir satırın hemen üzerinde, o satırın görevini özetleyen kısa bir `#` notu bulunur.

Bu notebook yalnızca **feature üretimi ve feature kalite kontrolü** yapar.

Girdi olarak Lipinski filtreli CSV değil, Notebook 01 tarafından oluşturulan
**temiz fakat filtrelenmemiş CSV** kullanılır.

Feature temizleme kuralları:

- Eksik oranı `%20` üzerinde olan descriptor çıkarılır.
- Bütün veri noktalarında aynı değere sahip sabit descriptor çıkarılır.
- Mutlak Pearson korelasyonu `0.80` üzerinde olan descriptor çiftlerinden biri çıkarılır.
- Model girdisi olarak yalnızca 2D Mordred descriptorları kullanılır.

Eksik kalan hücreler bu notebookta doldurulmaz. Median imputer yalnızca
Notebook 03'te eğitim setinde fit edilir.

## 1 — Paketler

Bu bölüm, notebookun ihtiyaç duyduğu Python paketlerini kontrol eder. Eksik paketler yalnızca gerektiğinde kurulur; böylece aynı kurulum komutları gereksiz yere tekrar çalıştırılmaz. Hücre tamamlandığında sonraki bölümlerde kullanılacak temel yazılımlar hazır olur.

In [ ]:
# importlib.util paketini kullanılabilir hâle getirir.
import importlib.util
# subprocess paketini kullanılabilir hâle getirir.
import subprocess
# sys paketini kullanılabilir hâle getirir.
import sys

# `PACKAGES` değişkenine bu adımda kullanılacak değeri atar.
PACKAGES = [
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("numpy", "numpy"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("pandas", "pandas"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("requests", "requests"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("rdkit", "rdkit"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("mordred", "mordredcommunity[full]==2.0.7"),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for import_name, pip_name in PACKAGES:
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if importlib.util.find_spec(import_name) is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        subprocess.check_call(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Paket kontrolü tamamlandı.")


## 2 — Kütüphaneler ve ayarlar

Bu bölümde kullanılacak kütüphaneler belleğe alınır ve pipeline boyunca değişmeden kullanılacak temel ayarlar tanımlanır. Target ID, klasör yolları, dosya adları ve tekrar üretilebilirlik değerleri burada merkezi olarak tutulur.

In [ ]:
# pathlib modülünden Path bileşenlerini içe aktarır.
from pathlib import Path
# io modülünden BytesIO bileşenlerini içe aktarır.
from io import BytesIO
# json paketini kullanılabilir hâle getirir.
import json

# numpy as np paketini kullanılabilir hâle getirir.
import numpy as np
# pandas as pd paketini kullanılabilir hâle getirir.
import pandas as pd
# requests paketini kullanılabilir hâle getirir.
import requests

# google.colab modülünden drive bileşenlerini içe aktarır.
from google.colab import drive
# IPython.display modülünden display bileşenlerini içe aktarır.
from IPython.display import display

# rdkit modülünden Chem bileşenlerini içe aktarır.
from rdkit import Chem
# mordred modülünden Calculator, descriptors bileşenlerini içe aktarır.
from mordred import Calculator, descriptors

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
drive.mount("/content/drive", force_remount=False)

# `DRIVE_DIR` değişkenine bu adımda kullanılacak değeri atar.
DRIVE_DIR = Path(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "/content/drive/MyDrive/MOL_FET_regression_pipeline"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `CONFIG_PATH` değişkenine bu adımda kullanılacak değeri atar.
CONFIG_PATH = DRIVE_DIR / "pipeline_config.json"

# `DEFAULT_TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_TARGET_ID = "CHEMBL206"
# `DEFAULT_GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_GITHUB_RAW_BASE = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "https://raw.githubusercontent.com/"
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "MOL-FEST/MOL-FET_regression/main"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if CONFIG_PATH.exists():
    # JSON metnini Python sözlüğüne dönüştürür.
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_id": DEFAULT_TARGET_ID,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "github_raw_base": DEFAULT_GITHUB_RAW_BASE,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "clean_unfiltered_filename": f"{DEFAULT_TARGET_ID}_clean_unfiltered.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "feature_filename": f"{DEFAULT_TARGET_ID}_Mordred2D_features.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "manifest_filename": f"{DEFAULT_TARGET_ID}_Mordred2D_manifest.json",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }

# `TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
TARGET_ID = config["target_id"]
# `GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_RAW_BASE = config["github_raw_base"]
# `INPUT_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
INPUT_FILENAME = config["clean_unfiltered_filename"]
# `FEATURE_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
FEATURE_FILENAME = config["feature_filename"]
# `MANIFEST_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
MANIFEST_FILENAME = config["manifest_filename"]

# `MAX_MISSING_RATIO` değişkenine bu adımda kullanılacak değeri atar.
MAX_MISSING_RATIO = 0.20
# `CORRELATION_THRESHOLD` değişkenine bu adımda kullanılacak değeri atar.
CORRELATION_THRESHOLD = 0.80

# Gerekli klasörü mevcut değilse oluşturur.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)


## 3 — Drive öncelikli girdi

Bu bölüm veri kaynağını hazırlar. Pipeline önce Google Drive içindeki önceki notebook çıktısını arar; dosya bulunamazsa aynı dosyayı GitHub RAW bağlantısından okumayı dener. Böylece notebooklar hem ardışık Colab çalışmasına hem de GitHub üzerinden bağımsız kullanıma uygundur.

In [ ]:
# `load_csv` adlı yardımcı fonksiyonu tanımlar.
def load_csv(drive_filename, github_relative_path):
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = DRIVE_DIR / drive_filename

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
        print("Girdi kaynağı: Google Drive")
        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return pd.read_csv(drive_path, low_memory=False)

    # `github_url` değişkenine bu adımda kullanılacak değeri atar.
    github_url = f"{GITHUB_RAW_BASE}/{github_relative_path}"
    # Belirtilen internet adresine HTTP GET isteği gönderir.
    response = requests.get(github_url, timeout=120)
    # HTTP yanıtında hata kodu varsa işlemi açıklayıcı hatayla durdurur.
    response.raise_for_status()

    # İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
    print("Girdi kaynağı: GitHub")
    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return pd.read_csv(BytesIO(response.content), low_memory=False)


# `clean_df` değişkenine bu adımda kullanılacak değeri atar.
clean_df = load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    INPUT_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"data/{INPUT_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `required` değişkenine bu adımda kullanılacak değeri atar.
required = {"parent_smiles", "pStandard"}
# `missing` değişkenine bu adımda kullanılacak değeri atar.
missing = required.difference(clean_df.columns)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise KeyError(f"Eksik sütunlar: {sorted(missing)}")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Temiz ve filtrelenmemiş veri:", clean_df.shape)


## 4 — RDKit molekülleri

Bu bölüm pipeline'ın tek bir görevini adım adım gerçekleştirir. Kod satırlarının hemen üstündeki `#` açıklamaları, ilgili komutun ne yaptığını Python deneyimi olmayan katılımcıların takip edebileceği şekilde özetler.

In [ ]:
# `molecules` değişkenine bu adımda kullanılacak değeri atar.
molecules = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    Chem.MolFromSmiles(smiles)
    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for smiles in clean_df["parent_smiles"]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `valid_mask` değişkenine bu adımda kullanılacak değeri atar.
valid_mask = np.array([
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    molecule is not None for molecule in molecules
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
])

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not valid_mask.all():
    # DataFrame indeksini yeniden düzenler.
    clean_df = clean_df.loc[valid_mask].reset_index(drop=True)
    # `molecules` değişkenine bu adımda kullanılacak değeri atar.
    molecules = [
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        molecule for molecule in molecules
        # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
        if molecule is not None
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not molecules:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("Mordred hesabı için geçerli molekül yok.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Mordred hesaplanacak molekül:", len(molecules))


## 5 — Ham 2D Mordred descriptorları

Bu bölüm molekülleri sayısal 2D Mordred descriptorlarına dönüştürür. Yüksek eksik oranlı, bütün moleküllerde aynı olan ve mutlak korelasyonu 0.80'i aşan gereksiz featurelar çıkarılır. Eksik kalan hücrelerin imputasyonu burada değil, eğitim verisi ayrıldıktan sonra Notebook 03'te yapılır.

In [ ]:
# `calculator` değişkenine bu adımda kullanılacak değeri atar.
calculator = Calculator(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    descriptors,
    # `ignore_3D` değişkenine bu adımda kullanılacak değeri atar.
    ignore_3D=True,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `mordred_raw` değişkenine bu adımda kullanılacak değeri atar.
mordred_raw = calculator.pandas(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    molecules,
    # `nproc` değişkenine bu adımda kullanılacak değeri atar.
    nproc=1,
    # `quiet` değişkenine bu adımda kullanılacak değeri atar.
    quiet=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
mordred_raw.columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    str(column) for column in mordred_raw.columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `mordred_numeric` değişkenine bu adımda kullanılacak değeri atar.
mordred_numeric = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    mordred_raw
    # Belirtilen fonksiyonu tablonun ilgili satır veya sütunlarına uygular.
    .apply(pd.to_numeric, errors="coerce")
    # Belirtilen değerleri güvenli alternatif değerlerle değiştirir.
    .replace([np.inf, -np.inf], np.nan)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Ham Mordred matrisi:", mordred_numeric.shape)


## 6 — Eksik ve sabit descriptor filtresi

Bu bölüm molekülleri sayısal 2D Mordred descriptorlarına dönüştürür. Yüksek eksik oranlı, bütün moleküllerde aynı olan ve mutlak korelasyonu 0.80'i aşan gereksiz featurelar çıkarılır. Eksik kalan hücrelerin imputasyonu burada değil, eğitim verisi ayrıldıktan sonra Notebook 03'te yapılır.

In [ ]:
# Eksik veya geçerli değerleri belirlemek için mantıksal maske üretir.
missing_ratio = mordred_numeric.isna().mean()

# `high_missing_columns` değişkenine bu adımda kullanılacak değeri atar.
high_missing_columns = missing_ratio[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    missing_ratio > MAX_MISSING_RATIO
# Diziyi veya pandas nesnesini standart Python listesine dönüştürür.
].index.tolist()

# Belirtilen satır veya sütunları veri yapısından çıkarır.
X_filtered = mordred_numeric.drop(
    # `columns` değişkenine bu adımda kullanılacak değeri atar.
    columns=high_missing_columns
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
).copy()

# `constant_columns` değişkenine bu adımda kullanılacak değeri atar.
constant_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column
    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for column in X_filtered.columns
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if X_filtered[column].nunique(dropna=True) <= 1
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen satır veya sütunları veri yapısından çıkarır.
X_filtered = X_filtered.drop(
    # `columns` değişkenine bu adımda kullanılacak değeri atar.
    columns=constant_columns
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
).copy()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if X_filtered.empty:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("Eksik ve sabit feature filtresinden sonra feature kalmadı.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Yüksek eksik nedeniyle çıkarılan:", len(high_missing_columns))
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Sabit olduğu için çıkarılan:", len(constant_columns))


## 7 — 0.80 korelasyon filtresi

Bu bölüm pipeline'ın tek bir görevini adım adım gerçekleştirir. Kod satırlarının hemen üstündeki `#` açıklamaları, ilgili komutun ne yaptığını Python deneyimi olmayan katılımcıların takip edebileceği şekilde özetler.

In [ ]:
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
correlation_input = X_filtered.copy()

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for column in correlation_input.columns:
    # `median_value` değişkenine bu adımda kullanılacak değeri atar.
    median_value = correlation_input[column].median()
    # Eksik değerleri belirtilen yöntem veya değerle doldurur.
    correlation_input[column] = correlation_input[column].fillna(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        median_value
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Featurelar arasındaki korelasyon katsayılarını hesaplar.
correlation_matrix = correlation_input.corr().abs()

# `upper_triangle` değişkenine bu adımda kullanılacak değeri atar.
upper_triangle = correlation_matrix.where(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    np.triu(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        np.ones(correlation_matrix.shape),
        # `k` değişkenine bu adımda kullanılacak değeri atar.
        k=1,
    # Veriyi belirtilen veri tipine dönüştürür.
    ).astype(bool)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `high_correlation_columns` değişkenine bu adımda kullanılacak değeri atar.
high_correlation_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column
    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for column in upper_triangle.columns
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        upper_triangle[column] > CORRELATION_THRESHOLD
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ).any()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen satır veya sütunları veri yapısından çıkarır.
X_final = X_filtered.drop(
    # `columns` değişkenine bu adımda kullanılacak değeri atar.
    columns=high_correlation_columns
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
).copy()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if X_final.empty:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("Korelasyon filtresinden sonra feature kalmadı.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Korelasyon > 0.80 nedeniyle çıkarılan:", len(high_correlation_columns))
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Final feature sayısı:", X_final.shape[1])


## 8 — Model-ready feature store

Bu bölüm molekülleri sayısal 2D Mordred descriptorlarına dönüştürür. Yüksek eksik oranlı, bütün moleküllerde aynı olan ve mutlak korelasyonu 0.80'i aşan gereksiz featurelar çıkarılır. Eksik kalan hücrelerin imputasyonu burada değil, eğitim verisi ayrıldıktan sonra Notebook 03'te yapılır.

In [ ]:
# `metadata_candidates` değişkenine bu adımda kullanılacak değeri atar.
metadata_candidates = [
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "parent_chembl_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "parent_smiles",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_mean",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_std",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_range",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_measurements",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_assays",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "activity_conflict",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "MolWt",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "MolLogP",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "HBD",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "HBA",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Lipinski_violations",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "passes_Lipinski",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `metadata_columns` değişkenine bu adımda kullanılacak değeri atar.
metadata_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column for column in metadata_candidates
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if column in clean_df.columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `feature_store` değişkenine bu adımda kullanılacak değeri atar.
feature_store = pd.concat(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [
        # DataFrame indeksini yeniden düzenler.
        clean_df[metadata_columns].reset_index(drop=True),
        # DataFrame indeksini yeniden düzenler.
        X_final.reset_index(drop=True),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ],
    # `axis` değişkenine bu adımda kullanılacak değeri atar.
    axis=1,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook içinde okunabilir biçimde gösterir.
display(feature_store.head())
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Feature store:", feature_store.shape)


## 9 — Google Drive kayıtları

Bu bölüm veri kaynağını hazırlar. Pipeline önce Google Drive içindeki önceki notebook çıktısını arar; dosya bulunamazsa aynı dosyayı GitHub RAW bağlantısından okumayı dener. Böylece notebooklar hem ardışık Colab çalışmasına hem de GitHub üzerinden bağımsız kullanıma uygundur.

In [ ]:
# `feature_path` değişkenine bu adımda kullanılacak değeri atar.
feature_path = DRIVE_DIR / FEATURE_FILENAME
# `manifest_path` değişkenine bu adımda kullanılacak değeri atar.
manifest_path = DRIVE_DIR / MANIFEST_FILENAME
# `raw_path` değişkenine bu adımda kullanılacak değeri atar.
raw_path = DRIVE_DIR / f"{TARGET_ID}_Mordred2D_raw.csv"

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
feature_store.to_csv(feature_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
mordred_numeric.to_csv(raw_path, index=False)

# `manifest` değişkenine bu adımda kullanılacak değeri atar.
manifest = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_id": TARGET_ID,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "input_filename": INPUT_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "feature_filename": FEATURE_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_molecules": int(len(feature_store)),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "raw_feature_count": int(mordred_numeric.shape[1]),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "final_feature_count": int(X_final.shape[1]),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "max_missing_ratio": MAX_MISSING_RATIO,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "correlation_threshold": CORRELATION_THRESHOLD,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "dropped_high_missing": high_missing_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "dropped_constant": constant_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "dropped_high_correlation": high_correlation_columns,
    # Diziyi veya pandas nesnesini standart Python listesine dönüştürür.
    "final_feature_names": X_final.columns.tolist(),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# Metin içeriğini belirtilen dosyaya kaydeder.
manifest_path.write_text(
    # Python sözlüğünü JSON metnine dönüştürür.
    json.dumps(manifest, ensure_ascii=False, indent=2),
    # `encoding` değişkenine bu adımda kullanılacak değeri atar.
    encoding="utf-8",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedildi:", feature_path)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedildi:", manifest_path)
